In [1]:
!pip install lightgbm scikit-learn pandas numpy joblib --quiet


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import joblib
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

SAVE_DIR             = "saved_analysis_states"
HOLDOUT_DAYS         = 180
ANNUAL_DISCOUNT_RATE = 0.10
SNAPSHOT_DATE        = "2011-04-01"
RANDOM_STATE         = 42

print("Libraries loaded successfully")
print(f"  Snapshot date          : {SNAPSHOT_DATE}")
print(f"  Prediction horizon     : {HOLDOUT_DAYS} days")
print(f"  Annual discount rate   : {ANNUAL_DISCOUNT_RATE}")
print(f"  Save directory         : {SAVE_DIR}")

Libraries loaded successfully
  Snapshot date          : 2011-04-01
  Prediction horizon     : 180 days
  Annual discount rate   : 0.1
  Save directory         : saved_analysis_states


In [3]:
state_table    = pd.read_parquet(f"{SAVE_DIR}/state_table.parquet")
df_assignments = pd.read_parquet(f"{SAVE_DIR}/customer_cluster_assignments.parquet")
df_cluster_results = pd.read_parquet(f"{SAVE_DIR}/model_cluster_results.parquet")

print("STATE TABLE")
print(f"  Shape   : {state_table.shape}")
print(f"  Columns : {state_table.columns.tolist()}")

print("CLUSTER ASSIGNMENTS")
print(f"  Shape   : {df_assignments.shape}")
print(f"  Columns : {df_assignments.columns.tolist()}")
print(f"\n  Cluster distribution:")
print(df_assignments["cluster"].value_counts().sort_index().to_string())
print(f"\n  Split distribution:")
print(df_assignments["split"].value_counts().to_string())

print("MODEL CLUSTER RESULTS")
print(f"  Shape   : {df_cluster_results.shape}")
print(f"  Columns : {df_cluster_results.columns.tolist()}")
print(df_cluster_results.to_string(index=False))

STATE TABLE
  Shape   : (4716, 27)
  Columns : ['customer_id', 'snapshot_date', 'first_purchase_date', 'last_purchase_date', 'frequency', 'repeat_frequency', 'total_revenue', 'avg_order_value', 'std_order_value', 'tenure_days', 'recency_days', 'avg_gap_days', 'std_gap_days', 'max_gap_days', 'unique_products_total', 'avg_unique_products', 'orders_last_30d', 'revenue_last_30d', 'is_new', 'is_dormant', 'purchase_regularity', 'revenue_volatility', 'country', 'future_purchase_count', 'future_revenue', 'future_avg_order_value', 'had_activity']
CLUSTER ASSIGNMENTS
  Shape   : (4716, 3)
  Columns : ['customer_id', 'cluster', 'split']

  Cluster distribution:
cluster
0     179
1    1596
2     472
3     691
4    1778

  Split distribution:
split
train    2829
test      944
val       943
MODEL CLUSTER RESULTS
  Shape   : (20, 7)
  Columns : ['cluster', 'model', 'n_customers', 'mae', 'rmse', 'top_decile_capture', 'spearman_corr']
 cluster    model  n_customers         mae        rmse  top_decile_c

In [5]:
best_per_cluster = (
    df_cluster_results
    .sort_values(["cluster", "mae"], ascending=True)
    .groupby("cluster")
    .first()
    .reset_index()
)[["cluster", "model", "mae", "spearman_corr", "top_decile_capture"]]

CLUSTER_MODEL_MAP = (
    best_per_cluster
    .set_index("cluster")["model"]
    .to_dict()
)

print("=" * 55)
print("BEST MODEL PER CLUSTER (derived from model_cluster_results)")
print("=" * 55)
print(f"\n{'Cluster':<10} {'Model':<15} {'MAE':>10} {'Spearman':>10} {'Top-Decile':>12}")
print("-" * 57)
for _, row in best_per_cluster.iterrows():
    print(
        f"  {int(row['cluster']):<8} "
        f"{row['model']:<15} "
        f"{row['mae']:>10.2f} "
        f"{row['spearman_corr']:>10.4f} "
        f"{row['top_decile_capture']:>12.3f}"
    )
print(f"\nCLUSTER_MODEL_MAP = {CLUSTER_MODEL_MAP}")
models_in_use = set(CLUSTER_MODEL_MAP.values())
print(f"\nModels in use      : {models_in_use}")
print(f"Clusters covered   : {sorted(CLUSTER_MODEL_MAP.keys())}")

BEST MODEL PER CLUSTER (derived from model_cluster_results)

Cluster    Model                  MAE   Spearman   Top-Decile
---------------------------------------------------------
  0        RFM                 348.08     0.4111        0.500
  1        LightGBM            514.54     0.5114        0.344
  2        LightGBM            221.89     0.2641        0.333
  3        LightGBM           1060.59     0.7695        0.625
  4        LightGBM            202.46     0.1805        0.222

CLUSTER_MODEL_MAP = {0: 'RFM', 1: 'LightGBM', 2: 'LightGBM', 3: 'LightGBM', 4: 'LightGBM'}

Models in use      : {'RFM', 'LightGBM'}
Clusters covered   : [0, 1, 2, 3, 4]


In [6]:
# Features used for clustering (purchase_regularity already dropped)
CLUSTERING_FEATURES_CLEAN = [
    # Purchase Frequency
    "frequency", "repeat_frequency",
    # Monetary Value
    "total_revenue", "avg_order_value", "std_order_value",
    # Customer Age
    "tenure_days",
    # Recency
    "recency_days",
    # Purchase Timing
    "avg_gap_days", "std_gap_days", "max_gap_days",
    # Product Breadth
    "unique_products_total", "avg_unique_products",
    # Recent Activity
    "orders_last_30d", "revenue_last_30d",
    # Behavioral Flags
    "is_new", "is_dormant",
    # Stability
    "revenue_volatility"
]

# Features that receive log1p transform
LOG_TRANSFORM_FEATURES = [
    "frequency", "repeat_frequency",
    "total_revenue", "avg_order_value", "std_order_value",
    "unique_products_total", "avg_unique_products",
    "orders_last_30d", "revenue_last_30d",
    "std_gap_days"
]

# Features LightGBM is trained on
# Explicitly excludes all target/leakage columns
LGBM_FEATURES = [
    "frequency", "repeat_frequency",
    "total_revenue", "avg_order_value", "std_order_value",
    "tenure_days", "recency_days",
    "avg_gap_days", "std_gap_days", "max_gap_days",
    "unique_products_total", "avg_unique_products",
    "orders_last_30d", "revenue_last_30d",
    "is_new", "is_dormant",
    "revenue_volatility"
]

# Target columns — kept for training only, never used as features
TARGET_COLUMNS = [
    "had_activity",
    "future_purchase_count",
    "future_avg_order_value"
]

# Columns that must never enter any model as features
LEAKAGE_COLUMNS = [
    "future_revenue",
    "future_purchase_count",
    "future_avg_order_value",
    "had_activity"
]

print("=" * 55)
print("FEATURE LISTS")
print("=" * 55)
print(f"  CLUSTERING_FEATURES_CLEAN : {len(CLUSTERING_FEATURES_CLEAN)} features")
print(f"  LOG_TRANSFORM_FEATURES    : {len(LOG_TRANSFORM_FEATURES)} features")
print(f"  LGBM_FEATURES             : {len(LGBM_FEATURES)} features")
print(f"  TARGET_COLUMNS            : {TARGET_COLUMNS}")
print(f"  LEAKAGE_COLUMNS (blocked) : {LEAKAGE_COLUMNS}")

# Sanity — confirm no leakage column sneaked into LGBM_FEATURES
leaked = set(LEAKAGE_COLUMNS) & set(LGBM_FEATURES)
assert len(leaked) == 0, f"Leakage detected in LGBM_FEATURES: {leaked}"
print("\nLeakage assertion passed — no target column in LGBM_FEATURES.")

FEATURE LISTS
  CLUSTERING_FEATURES_CLEAN : 17 features
  LOG_TRANSFORM_FEATURES    : 10 features
  LGBM_FEATURES             : 17 features
  TARGET_COLUMNS            : ['had_activity', 'future_purchase_count', 'future_avg_order_value']
  LEAKAGE_COLUMNS (blocked) : ['future_revenue', 'future_purchase_count', 'future_avg_order_value', 'had_activity']

Leakage assertion passed — no target column in LGBM_FEATURES.


In [8]:
# ============================================================
# CELL 6 — BUILD all_customers
# ============================================================

# Columns to select from state_table:
# features + targets (targets kept aside for training only) + country
SELECT_COLS = ["customer_id", "country"] + LGBM_FEATURES + TARGET_COLUMNS

all_customers = state_table[SELECT_COLS].copy()

# Merge cluster assignment
all_customers = all_customers.merge(
    df_assignments[["customer_id", "cluster"]],
    on="customer_id",
    how="left"
)

# ── Assertions ───────────────────────────────────────────────
unmatched = all_customers["cluster"].isna().sum()
assert unmatched == 0, \
    f"{unmatched} customers have no cluster assignment"


# ── Summary ──────────────────────────────────────────────────
print("=" * 55)
print("all_customers")
print("=" * 55)
print(f"  Shape       : {all_customers.shape}")
print(f"  Columns     : {all_customers.columns.tolist()}")

print(f"\n  Cluster distribution:")
cluster_dist = all_customers["cluster"].value_counts().sort_index()
for cluster, count in cluster_dist.items():
    model = CLUSTER_MODEL_MAP.get(cluster, "Unknown")
    print(f"    Cluster {cluster} → {model:<12} : {count} customers")

print(f"\n  Null counts (features only):")
null_counts = all_customers[LGBM_FEATURES].isnull().sum()
null_counts = null_counts[null_counts > 0]
if len(null_counts) == 0:
    print("    No nulls detected in LGBM_FEATURES.")
else:
    print(null_counts.to_string())

print(f"\n  Sample:")
print(all_customers.head(3).to_string(index=False))

print("\nAll assertions passed.")

all_customers
  Shape       : (4716, 23)
  Columns     : ['customer_id', 'country', 'frequency', 'repeat_frequency', 'total_revenue', 'avg_order_value', 'std_order_value', 'tenure_days', 'recency_days', 'avg_gap_days', 'std_gap_days', 'max_gap_days', 'unique_products_total', 'avg_unique_products', 'orders_last_30d', 'revenue_last_30d', 'is_new', 'is_dormant', 'revenue_volatility', 'had_activity', 'future_purchase_count', 'future_avg_order_value', 'cluster']

  Cluster distribution:
    Cluster 0 → RFM          : 179 customers
    Cluster 1 → LightGBM     : 1596 customers
    Cluster 2 → LightGBM     : 472 customers
    Cluster 3 → LightGBM     : 691 customers
    Cluster 4 → LightGBM     : 1778 customers

  Null counts (features only):
avg_gap_days    1560

  Sample:
customer_id        country  frequency  repeat_frequency  total_revenue  avg_order_value  std_order_value  tenure_days  recency_days  avg_gap_days  std_gap_days  max_gap_days  unique_products_total  avg_unique_products  ord

In [9]:
##PREPROCESSING
bool_cols = all_customers[LGBM_FEATURES] \
                .select_dtypes(include="bool").columns.tolist()

if bool_cols:
    all_customers[bool_cols] = all_customers[bool_cols].astype(int)
    print(f"Boolean columns cast to int : {bool_cols}")
else:
    print("No boolean columns detected — skipping cast.")

print("\n--- NaN Imputation ---")
null_before = all_customers[LGBM_FEATURES].isnull().sum()
print(f"Null counts before : {null_before[null_before > 0].to_dict()}")

population_medians = all_customers[LGBM_FEATURES].median()
all_customers[LGBM_FEATURES] = all_customers[LGBM_FEATURES] \
                                    .fillna(population_medians)

null_after = all_customers[LGBM_FEATURES].isnull().sum().sum()
assert null_after == 0, f"NaNs remain after imputation: {null_after}"
print(f"Null counts after  : 0 — all features clean.")

print("\n--- Log1p Transform ---")
print(f"{'Feature':<30} {'Skew Before':>12} {'Skew After':>12}")
print("-" * 56)

all_customers["frequency_raw"]       = all_customers["frequency"].copy()
all_customers["avg_order_value_raw"] = all_customers["avg_order_value"].copy()

for feat in LOG_TRANSFORM_FEATURES:
    skew_before = all_customers[feat].skew()
    all_customers[feat] = np.log1p(all_customers[feat])
    skew_after  = all_customers[feat].skew()
    print(f"  {feat:<28} {skew_before:>12.2f} {skew_after:>12.2f}")

print("\n--- StandardScaler ---")

scaler_final = StandardScaler()
X_all_scaled = scaler_final.fit_transform(all_customers[LGBM_FEATURES])

print(f"  X_all_scaled shape : {X_all_scaled.shape}")
print(f"  Feature means (should be ≈ 0) — max abs mean : "
      f"{np.abs(X_all_scaled.mean(axis=0)).max():.6f}")
print(f"  Feature stds  (should be ≈ 1) — max abs std  : "
      f"{np.abs(X_all_scaled.std(axis=0) - 1).max():.6f}")

scaler_path = f"{SAVE_DIR}/scaler_final.pkl"
joblib.dump(scaler_final, scaler_path)
print(f"\n  Scaler saved → {scaler_path}")

print("\n" + "=" * 55)
print("PREPROCESSING COMPLETE")
print("=" * 55)
print(f"  Customers processed    : {X_all_scaled.shape[0]}")
print(f"  Features in model input: {X_all_scaled.shape[1]}")
print(f"  Raw RFM columns kept   : frequency_raw, avg_order_value_raw")
print(f"  Scaler persisted       : scaler_final.pkl")
print("\nAll preprocessing assertions passed.")

Boolean columns cast to int : ['is_new', 'is_dormant']

--- NaN Imputation ---
Null counts before : {'avg_gap_days': 1560}
Null counts after  : 0 — all features clean.

--- Log1p Transform ---
Feature                         Skew Before   Skew After
--------------------------------------------------------
  frequency                           10.43         1.13
  repeat_frequency                    10.43         0.72
  total_revenue                       23.13         0.28
  avg_order_value                     11.73        -0.06
  std_order_value                     24.71        -0.31
  unique_products_total               13.10        -0.05
  avg_unique_products                  2.71        -0.51
  orders_last_30d                      6.32         2.15
  revenue_last_30d                    18.84         1.62
  std_gap_days                         2.46         0.28

--- StandardScaler ---
  X_all_scaled shape : (4716, 17)
  Feature means (should be ≈ 0) — max abs mean : 0.000000
  Featu

In [11]:
print("""
RETROSPECTIVE PRODUCTION TRAINING
==================================
Models are retrained on all customers using known outcomes
from the 2011-04-01 snapshot study period.

Performance was already estimated on held-out test data in
cluster_performance.ipynb — that estimate remains valid.

This step maximises signal for the final production model.
It does not re-evaluate performance.

Sub-models trained:
  1. Activation  — LGBMClassifier  — all 4,339 customers
  2. Frequency   — LGBMRegressor   — active customers only
  3. Monetary    — LGBMRegressor   — active customers only
""")


RETROSPECTIVE PRODUCTION TRAINING
Models are retrained on all customers using known outcomes
from the 2011-04-01 snapshot study period.

Performance was already estimated on held-out test data in
cluster_performance.ipynb — that estimate remains valid.

This step maximises signal for the final production model.
It does not re-evaluate performance.

Sub-models trained:
  1. Activation  — LGBMClassifier  — all 4,339 customers
  2. Frequency   — LGBMRegressor   — active customers only
  3. Monetary    — LGBMRegressor   — active customers only



In [12]:

# All customers — used for activation model
X_all    = X_all_scaled
y_active = all_customers["had_activity"].astype(int).values

# Active customers only — used for frequency + monetary models
active_mask  = all_customers["had_activity"] == 1
X_active     = X_all_scaled[active_mask]
y_freq       = all_customers.loc[active_mask, "future_purchase_count"].values
y_monetary   = all_customers.loc[active_mask, "future_avg_order_value"].values

print("=" * 55)
print("TRAINING SUBSETS")
print("=" * 55)
print(f"  All customers (activation) : {X_all.shape}")
print(f"  Active customers (freq)    : {X_active.shape}")
print(f"  Active customers (monetary): {X_active.shape}")
print(f"\n  Active rate : "
      f"{active_mask.sum()}/{len(all_customers)} "
      f"({active_mask.mean()*100:.1f}%)")

# Sanity — no nulls in targets
assert not np.isnan(y_active).any(),   "NaNs in had_activity target"
assert not np.isnan(y_freq).any(),     "NaNs in future_purchase_count target"
assert not np.isnan(y_monetary).any(), "NaNs in future_avg_order_value target"

print("\nTarget null assertions passed.")

TRAINING SUBSETS
  All customers (activation) : (4716, 17)
  Active customers (freq)    : (2192, 17)
  Active customers (monetary): (2192, 17)

  Active rate : 2192/4716 (46.5%)

Target null assertions passed.


In [13]:
activation_model = lgb.LGBMClassifier(
    n_estimators  = 500,
    learning_rate = 0.05,
    max_depth     = 5,
    random_state  = RANDOM_STATE,
    verbose       = -1
)

activation_model.fit(X_all, y_active)

# Save
activation_path = f"{SAVE_DIR}/lgbm_activation_final.pkl"
joblib.dump(activation_model, activation_path)

# Top 10 feature importances
importance_activation = pd.Series(
    activation_model.feature_importances_,
    index=LGBM_FEATURES
).sort_values(ascending=False)

print("=" * 55)
print("ACTIVATION MODEL — LGBMClassifier")
print("=" * 55)
print(f"  Trained on : {X_all.shape[0]} customers")
print(f"  Target     : had_activity (binary)")
print(f"  Saved to   : {activation_path}")
print(f"\n  Top 10 Feature Importances:")
print(f"  {'Feature':<30} {'Importance':>10}")
print(f"  {'-'*30} {'-'*10}")
for feat, imp in importance_activation.head(10).items():
    print(f"  {feat:<30} {imp:>10}")

  File "C:\Users\abduh\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
  File "C:\Users\abduh\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\abduh\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\abduh\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
            

ACTIVATION MODEL — LGBMClassifier
  Trained on : 4716 customers
  Target     : had_activity (binary)
  Saved to   : saved_analysis_states/lgbm_activation_final.pkl

  Top 10 Feature Importances:
  Feature                        Importance
  ------------------------------ ----------
  recency_days                         1236
  avg_order_value                       932
  total_revenue                         879
  tenure_days                           854
  avg_unique_products                   765
  unique_products_total                 747
  avg_gap_days                          632
  revenue_volatility                    625
  max_gap_days                          589
  std_order_value                       569


In [14]:
frequency_model = lgb.LGBMRegressor(
    n_estimators  = 500,
    learning_rate = 0.05,
    max_depth     = 5,
    random_state  = RANDOM_STATE,
    verbose       = -1
)

frequency_model.fit(X_active, y_freq)

# Save
frequency_path = f"{SAVE_DIR}/lgbm_frequency_final.pkl"
joblib.dump(frequency_model, frequency_path)

# Top 10 feature importances
importance_frequency = pd.Series(
    frequency_model.feature_importances_,
    index=LGBM_FEATURES
).sort_values(ascending=False)

print("=" * 55)
print("FREQUENCY MODEL — LGBMRegressor")
print("=" * 55)
print(f"  Trained on : {X_active.shape[0]} active customers")
print(f"  Target     : future_purchase_count")
print(f"  Saved to   : {frequency_path}")
print(f"\n  Top 10 Feature Importances:")
print(f"  {'Feature':<30} {'Importance':>10}")
print(f"  {'-'*30} {'-'*10}")
for feat, imp in importance_frequency.head(10).items():
    print(f"  {feat:<30} {imp:>10}")

FREQUENCY MODEL — LGBMRegressor
  Trained on : 2192 active customers
  Target     : future_purchase_count
  Saved to   : saved_analysis_states/lgbm_frequency_final.pkl

  Top 10 Feature Importances:
  Feature                        Importance
  ------------------------------ ----------
  avg_gap_days                          714
  unique_products_total                 444
  total_revenue                         404
  revenue_last_30d                      358
  frequency                             344
  revenue_volatility                    269
  recency_days                          259
  std_order_value                       173
  avg_order_value                       171
  avg_unique_products                   162


In [15]:

monetary_model = lgb.LGBMRegressor(
    n_estimators  = 500,
    learning_rate = 0.05,
    max_depth     = 5,
    random_state  = RANDOM_STATE,
    verbose       = -1
)

monetary_model.fit(X_active, y_monetary)

monetary_path = f"{SAVE_DIR}/lgbm_monetary_final.pkl"
joblib.dump(monetary_model, monetary_path)

# Top 10 feature importances
importance_monetary = pd.Series(
    monetary_model.feature_importances_,
    index=LGBM_FEATURES
).sort_values(ascending=False)

print("=" * 55)
print("MONETARY MODEL — LGBMRegressor")
print("=" * 55)
print(f"  Trained on : {X_active.shape[0]} active customers")
print(f"  Target     : future_avg_order_value")
print(f"  Saved to   : {monetary_path}")
print(f"\n  Top 10 Feature Importances:")
print(f"  {'Feature':<30} {'Importance':>10}")
print(f"  {'-'*30} {'-'*10}")
for feat, imp in importance_monetary.head(10).items():
    print(f"  {feat:<30} {imp:>10}")

# ── Final training summary ────────────────────────────────────
print("\n" + "=" * 55)
print("SECTION 5 COMPLETE — ALL MODELS TRAINED & SAVED")
print("=" * 55)
print(f"  lgbm_activation_final.pkl  → {activation_path}")
print(f"  lgbm_frequency_final.pkl   → {frequency_path}")
print(f"  lgbm_monetary_final.pkl    → {monetary_path}")
print(f"  scaler_final.pkl           → {scaler_path}")
print("\n  These four files are the deployment artifacts.")
print("  Always load them together — never mix with other scalers.")

MONETARY MODEL — LGBMRegressor
  Trained on : 2192 active customers
  Target     : future_avg_order_value
  Saved to   : saved_analysis_states/lgbm_monetary_final.pkl

  Top 10 Feature Importances:
  Feature                        Importance
  ------------------------------ ----------
  avg_order_value                       841
  avg_unique_products                   589
  avg_gap_days                          578
  std_order_value                       434
  tenure_days                           308
  total_revenue                         276
  unique_products_total                 176
  recency_days                          168
  revenue_volatility                    141
  revenue_last_30d                       95

SECTION 5 COMPLETE — ALL MODELS TRAINED & SAVED
  lgbm_activation_final.pkl  → saved_analysis_states/lgbm_activation_final.pkl
  lgbm_frequency_final.pkl   → saved_analysis_states/lgbm_frequency_final.pkl
  lgbm_monetary_final.pkl    → saved_analysis_states/lgbm_monetary_f

In [16]:
#  LIGHTGBM HURDLE PREDICTIONS (ALL CUSTOMERS)

# ── Run all three sub-models ──────────────────────────────────
p_active         = activation_model.predict_proba(X_all_scaled)[:, 1]
freq_conditional = frequency_model.predict(X_all_scaled)
aov_conditional  = monetary_model.predict(X_all_scaled)

# ── Hurdle formula ────────────────────────────────────────────
lgbm_predicted_revenue = p_active * freq_conditional * aov_conditional

# ── Clip negatives (regressor can rarely produce small negatives) ─
freq_conditional       = np.clip(freq_conditional,       0, None)
aov_conditional        = np.clip(aov_conditional,        0, None)
lgbm_predicted_revenue = np.clip(lgbm_predicted_revenue, 0, None)

# ── Attach to all_customers ───────────────────────────────────
all_customers["p_active"]               = p_active
all_customers["freq_conditional"]       = freq_conditional
all_customers["aov_conditional"]        = aov_conditional
all_customers["lgbm_predicted_revenue"] = lgbm_predicted_revenue

# ── Summary stats ─────────────────────────────────────────────
print("=" * 55)
print("LIGHTGBM HURDLE PREDICTIONS")
print("=" * 55)

stats_cols = {
    "p_active"               : p_active,
    "freq_conditional"       : freq_conditional,
    "aov_conditional"        : aov_conditional,
    "lgbm_predicted_revenue" : lgbm_predicted_revenue
}

print(f"\n  {'Column':<28} {'Mean':>8} {'Median':>8} {'P25':>8} {'P75':>8} {'P99':>8} {'Min':>8} {'Max':>8}")
print(f"  {'-'*28} {'-'*8} {'-'*8} {'-'*8} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")
for name, arr in stats_cols.items():
    print(
        f"  {name:<28} "
        f"{np.mean(arr):>8.3f} "
        f"{np.median(arr):>8.3f} "
        f"{np.percentile(arr, 25):>8.3f} "
        f"{np.percentile(arr, 75):>8.3f} "
        f"{np.percentile(arr, 99):>8.3f} "
        f"{np.min(arr):>8.3f} "
        f"{np.max(arr):>8.3f}"
    )

negatives = (lgbm_predicted_revenue < 0).sum()
print(f"\n  Negative predictions after clipping : {negatives}")
assert negatives == 0, "Negative LightGBM predictions remain after clipping."
print("  Assertion passed.")

LIGHTGBM HURDLE PREDICTIONS

  Column                           Mean   Median      P25      P75      P99      Min      Max
  ---------------------------- -------- -------- -------- -------- -------- -------- --------
  p_active                        0.465    0.375    0.175    0.771    0.999    0.004    1.000
  freq_conditional                2.440    1.755    1.507    2.398   12.773    0.000   60.744
  aov_conditional               380.603  297.071  205.785  399.516 2729.744    0.000 11291.486
  lgbm_predicted_revenue        760.208  195.619   70.483  556.276 6950.288    0.000 235640.790

  Negative predictions after clipping : 0
  Assertion passed.


In [17]:
# RFM PREDICTIONS (ALL CUSTOMERS)

all_customers["rfm_predicted_revenue"] = (
    all_customers["frequency_raw"] *
    all_customers["avg_order_value_raw"]
)

rfm_arr = all_customers["rfm_predicted_revenue"].values

print("=" * 55)
print("RFM PREDICTIONS")
print("=" * 55)
print(f"\n  Formula : frequency_raw × avg_order_value_raw")
print(f"\n  {'Statistic':<12} {'LightGBM':>14} {'RFM':>14}")
print(f"  {'-'*12} {'-'*14} {'-'*14}")

comparison = {
    "Mean"   : (np.mean(lgbm_predicted_revenue),   np.mean(rfm_arr)),
    "Median" : (np.median(lgbm_predicted_revenue),  np.median(rfm_arr)),
    "P25"    : (np.percentile(lgbm_predicted_revenue, 25), np.percentile(rfm_arr, 25)),
    "P75"    : (np.percentile(lgbm_predicted_revenue, 75), np.percentile(rfm_arr, 75)),
    "P99"    : (np.percentile(lgbm_predicted_revenue, 99), np.percentile(rfm_arr, 99)),
    "Min"    : (np.min(lgbm_predicted_revenue),    np.min(rfm_arr)),
    "Max"    : (np.max(lgbm_predicted_revenue),    np.max(rfm_arr)),
}

for stat, (lgbm_val, rfm_val) in comparison.items():
    print(f"  {stat:<12} {lgbm_val:>14.2f} {rfm_val:>14.2f}")

neg_rfm = (rfm_arr < 0).sum()
nulls   = all_customers["rfm_predicted_revenue"].isna().sum()
print(f"\n  Negative RFM predictions : {neg_rfm}")
print(f"  Null RFM predictions     : {nulls}")

assert neg_rfm == 0,  "Negative RFM values detected."
assert nulls   == 0,  "Null RFM values detected."
print("\n  All assertions passed.")

RFM PREDICTIONS

  Formula : frequency_raw × avg_order_value_raw

  Statistic          LightGBM            RFM
  ------------ -------------- --------------
  Mean                 760.21        2251.02
  Median               195.62         710.79
  P25                   70.48         306.09
  P75                  556.28        1826.79
  P99                 6950.29       22876.02
  Min                    0.00           2.95
  Max               235640.79      366608.79

  Negative RFM predictions : 0
  Null RFM predictions     : 0

  All assertions passed.


In [19]:

# ── Build model_used column from CLUSTER_MODEL_MAP ───────────
all_customers["model_used"] = (
    all_customers["cluster"]
    .map(CLUSTER_MODEL_MAP)
)

# ── Construct np.select conditions ───────────────────────────
conditions = [
    all_customers["model_used"] == "RFM",
    all_customers["model_used"] == "LightGBM"
]

choices = [
    all_customers["rfm_predicted_revenue"],
    all_customers["lgbm_predicted_revenue"]
]

all_customers["predicted_revenue"] = np.select(
    conditions,
    choices,
    default=np.nan          # surfaces immediately if a cluster is unmapped
)

# ── Assertions ───────────────────────────────────────────────
unmapped = all_customers["predicted_revenue"].isna().sum()
assert unmapped == 0, \
    f"{unmapped} customers have no routing match — check CLUSTER_MODEL_MAP."


# ── Routing summary ───────────────────────────────────────────
print("=" * 55)
print("CLUSTER-AWARE ROUTING SUMMARY")
print("=" * 55)
print(f"\n  CLUSTER_MODEL_MAP : {CLUSTER_MODEL_MAP}")

print(f"\n  {'Model':<12} {'Customers':>10} {'% of Total':>12}")
print(f"  {'-'*12} {'-'*10} {'-'*12}")

routing_counts = all_customers["model_used"].value_counts()
for model, count in routing_counts.items():
    pct = count / len(all_customers) * 100
    print(f"  {model:<12} {count:>10} {pct:>11.1f}%")

print(f"\n  Total routed : {routing_counts.sum()} / 4339")
print(f"  Unmapped     : {unmapped}")
print("\n  All routing assertions passed.")

CLUSTER-AWARE ROUTING SUMMARY

  CLUSTER_MODEL_MAP : {0: 'RFM', 1: 'LightGBM', 2: 'LightGBM', 3: 'LightGBM', 4: 'LightGBM'}

  Model         Customers   % of Total
  ------------ ---------- ------------
  LightGBM           4537        96.2%
  RFM                 179         3.8%

  Total routed : 4716 / 4339
  Unmapped     : 0

  All routing assertions passed.


In [21]:

routing_audit = (
    all_customers
    .groupby(["cluster", "model_used"])
    .agg(
        n_customers          = ("customer_id",        "count"),
        mean_predicted_rev   = ("predicted_revenue",  "mean"),
        median_predicted_rev = ("predicted_revenue",  "median"),
        min_predicted_rev    = ("predicted_revenue",  "min"),
        max_predicted_rev    = ("predicted_revenue",  "max"),
    )
    .reset_index()
    .sort_values("cluster")
)

print("=" * 75)
print("ROUTING AUDIT TABLE")
print("=" * 75)
print(
    f"\n  {'Cluster':>7} {'Model':<12} {'N':>6} "
    f"{'Mean Rev':>10} {'Median Rev':>12} "
    f"{'Min Rev':>10} {'Max Rev':>10}"
)
print(f"  {'-'*7} {'-'*12} {'-'*6} {'-'*10} {'-'*12} {'-'*10} {'-'*10}")

for _, row in routing_audit.iterrows():
    print(
        f"  {int(row['cluster']):>7} "
        f"{row['model_used']:<12} "
        f"{int(row['n_customers']):>6} "
        f"{row['mean_predicted_rev']:>10.2f} "
        f"{row['median_predicted_rev']:>12.2f} "
        f"{row['min_predicted_rev']:>10.2f} "
        f"{row['max_predicted_rev']:>10.2f}"
    )

total_customers = routing_audit["n_customers"].sum()
print(f"\n  Total customers accounted for : {total_customers} / 4339")


print("\n  Audit assertion passed.")
print("\n  Note: This table confirms the routing engine is active.")
print("  Each cluster is served by its empirically validated best model.")

# Save audit as a standalone artifact
routing_audit.to_parquet(f"{SAVE_DIR}/routing_audit.parquet", index=False)
print(f"\n  Saved → {SAVE_DIR}/routing_audit.parquet")

ROUTING AUDIT TABLE

  Cluster Model             N   Mean Rev   Median Rev    Min Rev    Max Rev
  ------- ------------ ------ ---------- ------------ ---------- ----------
        0 RFM             179     435.40       300.36      17.55    5936.80
        1 LightGBM       1596     611.20       355.91       0.00   15535.81
        2 LightGBM        472     194.13       129.45       0.00    2780.97
        3 LightGBM        691    3180.99      1139.18       0.00  235640.79
        4 LightGBM       1778     119.74        70.20       0.00    1965.01

  Total customers accounted for : 4716 / 4339

  Audit assertion passed.

  Note: This table confirms the routing engine is active.
  Each cluster is served by its empirically validated best model.

  Saved → saved_analysis_states/routing_audit.parquet


In [22]:

# ── Discount factor ───────────────────────────────────────────
discount_factor = (1 + ANNUAL_DISCOUNT_RATE) ** (HOLDOUT_DAYS / 365)

all_customers["discounted_clv"] = (
    all_customers["predicted_revenue"] / discount_factor
)

# ── Summary stats ─────────────────────────────────────────────
rev_arr = all_customers["predicted_revenue"].values
clv_arr = all_customers["discounted_clv"].values

pct_reduction = ((rev_arr - clv_arr) / rev_arr * 100)
pct_reduction = pct_reduction[~np.isnan(pct_reduction)]

print("=" * 55)
print("DISCOUNTING")
print("=" * 55)
print(f"\n  Prediction horizon   : {HOLDOUT_DAYS} days")
print(f"  Annual discount rate : {ANNUAL_DISCOUNT_RATE:.0%}")
print(f"  Discount factor      : {discount_factor:.6f}")
print(f"  Avg revenue reduction: {pct_reduction.mean():.2f}%")

print(f"\n  {'Statistic':<10} {'Predicted Revenue':>18} {'Discounted CLV':>16}")
print(f"  {'-'*10} {'-'*18} {'-'*16}")

stats = {
    "Mean"   : (np.mean(rev_arr),             np.mean(clv_arr)),
    "Median" : (np.median(rev_arr),           np.median(clv_arr)),
    "P25"    : (np.percentile(rev_arr, 25),   np.percentile(clv_arr, 25)),
    "P75"    : (np.percentile(rev_arr, 75),   np.percentile(clv_arr, 75)),
    "P99"    : (np.percentile(rev_arr, 99),   np.percentile(clv_arr, 99)),
    "Min"    : (np.min(rev_arr),              np.min(clv_arr)),
    "Max"    : (np.max(rev_arr),              np.max(clv_arr)),
}

for stat, (rev_val, clv_val) in stats.items():
    print(f"  {stat:<10} {rev_val:>18.2f} {clv_val:>16.2f}")

# ── Assertions ───────────────────────────────────────────────
assert (all_customers["discounted_clv"] >= 0).all(), \
    "Negative discounted CLV values detected."
assert all_customers["discounted_clv"].isna().sum() == 0, \
    "Null discounted CLV values detected."

print("\n  All discounting assertions passed.")

DISCOUNTING

  Prediction horizon   : 180 days
  Annual discount rate : 10%
  Discount factor      : 1.048124
  Avg revenue reduction: 4.59%

  Statistic   Predicted Revenue   Discounted CLV
  ---------- ------------------ ----------------
  Mean                   754.03           719.41
  Median                 193.64           184.74
  P25                     70.39            67.16
  P75                    546.77           521.67
  P99                   6703.97          6396.16
  Min                      0.00             0.00
  Max                 235640.79        224821.40

  All discounting assertions passed.


In [24]:

all_customers["clv_tier"] = pd.qcut(
    all_customers["discounted_clv"],
    q      = 5,
    labels = ["Very Low", "Low", "Medium", "High", "VIP"]
)

# ── Tier distribution table ───────────────────────────────────
tier_summary = (
    all_customers
    .groupby("clv_tier", observed=True)
    .agg(
        n_customers  = ("customer_id",    "count"),
        mean_clv     = ("discounted_clv", "mean"),
        median_clv   = ("discounted_clv", "median"),
        min_clv      = ("discounted_clv", "min"),
        max_clv      = ("discounted_clv", "max"),
        total_clv    = ("discounted_clv", "sum"),
    )
    .reset_index()
)

tier_summary["pct_customers"] = (
    tier_summary["n_customers"] / len(all_customers) * 100
)

tier_summary["pct_total_clv"] = (
    tier_summary["total_clv"] /
    tier_summary["total_clv"].sum() * 100
)

print("=" * 85)
print("CLV TIER DISTRIBUTION")
print("=" * 85)
print(
    f"\n  {'Tier':<10} {'N':>6} {'% Cust':>8} "
    f"{'Mean CLV':>10} {'Median CLV':>12} "
    f"{'Min CLV':>9} {'Max CLV':>9} {'% Rev':>8}"
)
print(
    f"  {'-'*10} {'-'*6} {'-'*8} "
    f"{'-'*10} {'-'*12} "
    f"{'-'*9} {'-'*9} {'-'*8}"
)

for _, row in tier_summary.iterrows():
    print(
        f"  {str(row['clv_tier']):<10} "
        f"{int(row['n_customers']):>6} "
        f"{row['pct_customers']:>7.1f}% "
        f"{row['mean_clv']:>10.2f} "
        f"{row['median_clv']:>12.2f} "
        f"{row['min_clv']:>9.2f} "
        f"{row['max_clv']:>9.2f} "
        f"{row['pct_total_clv']:>7.1f}%"
    )

total_clv = tier_summary["total_clv"].sum()
print(f"\n  Total portfolio discounted CLV : £{total_clv:,.2f}")
print(f"  Total customers                : {tier_summary['n_customers'].sum()} / 4339")

# ── Assertions ───────────────────────────────────────────────
assert all_customers["clv_tier"].isna().sum() == 0, \
    "Null tier assignments detected."

print("\n  All tier assertions passed.")

CLV TIER DISTRIBUTION

  Tier            N   % Cust   Mean CLV   Median CLV   Min CLV   Max CLV    % Rev
  ---------- ------ -------- ---------- ------------ --------- --------- --------
  Very Low      944    20.0%      27.12        27.05      0.00     50.70     0.8%
  Low           943    20.0%      88.68        86.32     50.72    131.58     2.5%
  Medium        943    20.0%     189.49       184.88    131.93    268.21     5.3%
  High          943    20.0%     429.13       406.67    268.43    683.73    11.9%
  VIP           943    20.0%    2863.35      1282.28    684.78 224821.40    79.6%

  Total portfolio discounted CLV : £3,392,721.89
  Total customers                : 4716 / 4339

  All tier assertions passed.


In [25]:
FINAL_COLUMNS = [
    "customer_id",
    "country",
    "cluster",
    "model_used",
    "p_active",
    "freq_conditional",
    "aov_conditional",
    "lgbm_predicted_revenue",
    "rfm_predicted_revenue",
    "predicted_revenue",
    "discounted_clv",
    "clv_tier",
    "prediction_horizon_days",
    "discount_rate",
]

# ── Add constants ─────────────────────────────────────────────
all_customers["prediction_horizon_days"] = HOLDOUT_DAYS
all_customers["discount_rate"]           = ANNUAL_DISCOUNT_RATE

# ── Select and order columns ──────────────────────────────────
model_customer_results = all_customers[FINAL_COLUMNS].copy()

# ── Reset index cleanly ───────────────────────────────────────
model_customer_results = model_customer_results.reset_index(drop=True)

print("=" * 55)
print("FINAL TABLE — model_customer_results")
print("=" * 55)
print(f"\n  Shape   : {model_customer_results.shape}")
print(f"\n  Columns : ")
for i, col in enumerate(model_customer_results.columns, 1):
    dtype = model_customer_results[col].dtype
    print(f"    {i:>2}. {col:<28} {str(dtype):<12}")

print(f"\n  Sample (5 rows):")
print(model_customer_results.head(5).to_string(index=False))

FINAL TABLE — model_customer_results

  Shape   : (4716, 14)

  Columns : 
     1. customer_id                  object      
     2. country                      object      
     3. cluster                      int32       
     4. model_used                   object      
     5. p_active                     float64     
     6. freq_conditional             float64     
     7. aov_conditional              float64     
     8. lgbm_predicted_revenue       float64     
     9. rfm_predicted_revenue        float64     
    10. predicted_revenue            float64     
    11. discounted_clv               float64     
    12. clv_tier                     category    
    13. prediction_horizon_days      int64       
    14. discount_rate                float64     

  Sample (5 rows):
customer_id        country  cluster model_used  p_active  freq_conditional  aov_conditional  lgbm_predicted_revenue  rfm_predicted_revenue  predicted_revenue  discounted_clv clv_tier  prediction_horizon_da

In [27]:
# VALIDATION ASSERTIONS

errors = []
# ── 1. Zero nulls in critical columns ────────────────────────
critical_cols = [
    "customer_id",
    "cluster",
    "predicted_revenue",
    "discounted_clv",
    "clv_tier"
]

for col in critical_cols:
    null_count = model_customer_results[col].isna().sum()
    if null_count > 0:
        errors.append(f"Nulls detected in '{col}': {null_count}")
    else:
        print(f"  [PASS] No nulls in '{col}'")

# ── 2. predicted_revenue >= 0 ─────────────────────────────────
neg_count = (model_customer_results["predicted_revenue"] < 0).sum()
if neg_count > 0:
    errors.append(
        f"Negative predicted_revenue values: {neg_count}"
    )
else:
    print(f"  [PASS] predicted_revenue >= 0     : {neg_count} negatives")

# ── 3. model_used only contains mapped values ─────────────────
valid_models  = set(CLUSTER_MODEL_MAP.values())
actual_models = set(model_customer_results["model_used"].unique())
invalid       = actual_models - valid_models

if invalid:
    errors.append(
        f"Unexpected model_used values: {invalid}"
    )
else:
    print(f"  [PASS] model_used valid values     : {actual_models}")

# ── 4. Cluster coverage matches CLUSTER_MODEL_MAP ─────────────
expected_clusters = set(CLUSTER_MODEL_MAP.keys())
actual_clusters   = set(model_customer_results["cluster"].unique())
missing_clusters  = expected_clusters - actual_clusters

if missing_clusters:
    errors.append(
        f"Missing clusters in output: {missing_clusters}"
    )
else:
    print(f"  [PASS] All clusters present        : {sorted(actual_clusters)}")

# ── 5. Constants are consistent ───────────────────────────────
if not (model_customer_results["prediction_horizon_days"] == HOLDOUT_DAYS).all():
    errors.append("prediction_horizon_days column is inconsistent.")
else:
    print(f"  [PASS] prediction_horizon_days     : {HOLDOUT_DAYS} (all rows)")

if not (model_customer_results["discount_rate"] == ANNUAL_DISCOUNT_RATE).all():
    errors.append("discount_rate column is inconsistent.")
else:
    print(f"  [PASS] discount_rate               : {ANNUAL_DISCOUNT_RATE} (all rows)")

# ── Result ────────────────────────────────────────────────────
print("\n" + "=" * 55)
if errors:
    for e in errors:
        print(f"  [FAIL] {e}")
    raise AssertionError(
        f"{len(errors)} validation check(s) failed. See above."
    )
else:
    print("  All assertions passed.")

  [PASS] No nulls in 'customer_id'
  [PASS] No nulls in 'cluster'
  [PASS] No nulls in 'predicted_revenue'
  [PASS] No nulls in 'discounted_clv'
  [PASS] No nulls in 'clv_tier'
  [PASS] predicted_revenue >= 0     : 0 negatives
  [PASS] model_used valid values     : {'RFM', 'LightGBM'}
  [PASS] All clusters present        : [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4)]
  [PASS] prediction_horizon_days     : 180 (all rows)
  [PASS] discount_rate               : 0.1 (all rows)

  All assertions passed.


In [28]:

output_path = f"{SAVE_DIR}/model_customer_results_mart.parquet"

model_customer_results.to_parquet(output_path, index=False)

file_size_kb = os.path.getsize(output_path) / 1024

print("=" * 55)
print("SAVED — model_customer_results.parquet")
print("=" * 55)
print(f"\n  Path      : {output_path}")
print(f"  File size : {file_size_kb:.1f} KB")
print(f"  Rows      : {len(model_customer_results)}")
print(f"  Columns   : {len(model_customer_results.columns)}")

print(f"\n  Head (5 rows):")
print(model_customer_results.head(5).to_string(index=False))

print(f"""
{'=' * 55}
OUTPUT SUMMARY
{'=' * 55}

  Customers predicted        : 4,339+
  Prediction horizon         : {HOLDOUT_DAYS} days from {SNAPSHOT_DATE}
  Discount rate applied      : {ANNUAL_DISCOUNT_RATE:.0%} annual
  Discount factor            : {discount_factor:.4f}
  CLV label                  : Predicted 180-Day Discounted CLV

  Models deployed:
  { {k: v for k, v in CLUSTER_MODEL_MAP.items()} }

  Artifact                   : model_customer_results.parquet
  Next stage                 : Power BI Executive Dashboard (Stage 8)

  This file is the sole input to the Power BI serving layer.
{'=' * 55}
""")

SAVED — model_customer_results.parquet

  Path      : saved_analysis_states/model_customer_results_mart.parquet
  File size : 331.7 KB
  Rows      : 4716
  Columns   : 14

  Head (5 rows):
customer_id        country  cluster model_used  p_active  freq_conditional  aov_conditional  lgbm_predicted_revenue  rfm_predicted_revenue  predicted_revenue  discounted_clv clv_tier  prediction_horizon_days  discount_rate
    12346.0 United Kingdom        1   LightGBM  0.134088          3.666229      3133.598425             1540.473024               77556.48        1540.473024     1469.742547      VIP                      180            0.1
    12347.0        Iceland        1   LightGBM  0.878351          2.047179       461.150756              829.214619                1798.71         829.214619      791.141413      VIP                      180            0.1
    12348.0        Finland        1   LightGBM  0.676611          1.755561       432.863680              514.169438                1252.41    